In [87]:
import pandas as pd
from collections import Counter
import os
import sys

In [88]:
sys.path.append(os.path.abspath(".."))
import src.analysis_utils as au

## Data Load

In [89]:
df = pd.read_csv("../data/raw/reviews_nlp.csv")
df

,id,avis,label,categorie,difficulte
0,1,Le service client a été très rapide et très pr...,positif,support,simple
1,2,"Je suis très déçu, ma commande est arrivée cas...",négatif,produit,simple
2,3,Votre demande a été enregistrée.,neutre,administratif,simple
3,4,Le produit est bon mais la livraison a été bea...,mitigé,livraison,opinion_mixte
4,5,"Super, encore une panne de l'application.",négatif,application,sarcasme
...,...,...,...,...,...
995,996,Je ne m'attendais pas à autant de satisfaction...,positif,general,négation_positive
996,997,"La livraison est fiable, rapide et soignée.",positif,livraison,simple
997,998,Je suis très satisfait de votre service dans l...,positif,general,simple
998,999,"L'appli est bien conçue, intuitive et rapide.",positif,application,simple


## I. Exploration

In [90]:
print("\n=== Dimensions du corpus ===")
print(df.shape)

print("\n=== Colonnes ===")
print(df.columns.tolist())

print("\n=== Types ===")
print(df.dtypes)

print("\n=== Valeurs manquantes ===")
print(df.isna().sum())

print("\n=== Doublons ===")
print(df.duplicated().sum())


=== Dimensions du corpus ===
(1000, 5)

=== Colonnes ===
['id', 'avis', 'label', 'categorie', 'difficulte']

=== Types ===
id            int64
avis            str
label           str
categorie       str
difficulte      str
dtype: object

=== Valeurs manquantes ===
id            0
avis          0
label         0
categorie     0
difficulte    0
dtype: int64

=== Doublons ===
0


In [91]:
print("=== Répartition des sources ===")
print(df.groupby("categorie").size())

print("\n=== Répartition des labels ===")
print(df.groupby('label').size())

print("\n=== Répartition des difficultes ===")
print(df.groupby('difficulte').size())

=== Répartition des sources ===
categorie
administratif     46
application      142
general          206
livraison        152
paiement          28
produit          196
remboursement     48
support          182
dtype: int64

=== Répartition des labels ===
label
mitigé      85
neutre      56
négatif    185
positif    674
dtype: int64

=== Répartition des difficultes ===
difficulte
factuel               55
négation_positive    191
opinion_mixte         82
problème             155
sarcasme              20
simple               497
dtype: int64


In [92]:
df["text_length_chars"] = df["avis"].astype(str).str.len()
df["text_word_count"] = df["avis"].astype(str).str.split().str.len()

print("\n=== Statistiques de longueur en caractères ===")
print(df["text_length_chars"].describe())

print("\n=== Statistiques du nombre de mots ===")
print(df["text_word_count"].describe())

print("\n=== Textes les plus courts ===")
print(df.sort_values("text_word_count")[["id", "avis", "label", "text_word_count"]].head(5))


=== Statistiques de longueur en caractères ===
count    1000.000000
mean       45.657000
std        10.197026
min        15.000000
25%        38.000000
50%        45.000000
75%        53.000000
max        92.000000
Name: text_length_chars, dtype: float64

=== Statistiques du nombre de mots ===
count    1000.000000
mean        7.621000
std         2.070663
min         2.000000
25%         6.000000
50%         7.000000
75%         9.000000
max        15.000000
Name: text_word_count, dtype: float64

=== Textes les plus courts ===
      id                                avis    label  text_word_count
395  396                     Commande reçue.   neutre                2
31    32        Livraison express, parfaite.  positif                3
414  415  Confirmation d'expédition envoyée.   neutre                3
507  508        Remboursement reçu conforme.  positif                3
374  375          Bonne expérience générale.  positif                3


In [93]:
print("\n=== Textes les plus longs ===")
df.sort_values("text_word_count", ascending=False)[["id", "avis", "label", "text_word_count"]]


=== Textes les plus longs ===


,id,avis,label,text_word_count
283,284,Je ne pensais pas faire confiance à un service...,positif,15
821,822,Je suis très satisfait de la façon dont ma réc...,positif,15
80,81,Mon compte a été piraté et votre support n'a p...,négatif,14
922,923,Je ne pensais pas que votre service serait aus...,positif,14
469,470,Je n'aurais pas parié sur un service en ligne ...,positif,14
...,...,...,...,...
208,209,Montant débité incorrect.,négatif,3
31,32,"Livraison express, parfaite.",positif,3
507,508,Remboursement reçu conforme.,positif,3
111,112,Excellent rapport qualité-prix.,positif,3


In [94]:
all_tokens = []

for text in df["avis"]:
    all_tokens.extend(au.simple_tokenize(text))

token_counts = Counter(all_tokens)

n = 5
print(f"\n=== {n} mots les plus fréquents ===")
for token, count in token_counts.most_common(n):
    print(token, ":", count)


=== 5 mots les plus fréquents ===
je : 333
est : 277
de : 260
pas : 232
la : 204


In [95]:
print("\n=== Longueurs moyennes par label ===")
length_by_label = df.groupby("label")[["text_length_chars", "text_word_count"]].mean()
print(length_by_label.round(2))


=== Longueurs moyennes par label ===
         text_length_chars  text_word_count
label                                      
mitigé               50.99             8.25
neutre               34.80             5.54
négatif              47.07             7.77
positif              45.50             7.68


In [96]:
print("\n=== Mots fréquents par label ===")

for label in sorted(df["label"].unique()):
    label_tokens = []

    subset = df[df["label"] == label]

    for text in subset["avis"]:
        label_tokens.extend(au.simple_tokenize(text))

    counts = Counter(label_tokens)
    print(f"\nLabel : {label}")
    print(counts.most_common(10))


=== Mots fréquents par label ===

Label : mitigé
[('mais', 30), ('est', 28), ('le', 27), ('de', 23), ('pas', 15), ('je', 15), ('la', 14), ('livraison', 12), ('service', 12), ('à', 11)]

Label : neutre
[('de', 22), ('commande', 11), ('à', 11), ('jour', 9), ('livraison', 7), ('reçue', 6), ('en', 6), ('votre', 5), ('a', 5), ('été', 5)]

Label : négatif
[('de', 51), ('pas', 42), ('le', 39), ('à', 38), ('la', 30), ('votre', 28), ('je', 27), ('est', 26), ('ne', 26), ('mon', 25)]

Label : positif
[('je', 291), ('est', 219), ('pas', 175), ('service', 170), ('de', 164), ('la', 156), ('ne', 145), ('le', 124), ('à', 120), ('votre', 116)]


## II. Preprocessing

In [97]:
df["clean_avis"] = df["avis"].apply(au.clean_text)

print("\n=== Comparaison texte brut / texte nettoyé ===")
df[["id", "avis", "clean_avis"]]


=== Comparaison texte brut / texte nettoyé ===


,id,avis,clean_avis
0,1,Le service client a été très rapide et très pr...,le service client a été très rapide et très pr...
1,2,"Je suis très déçu, ma commande est arrivée cas...",je suis très déçu ma commande est arrivée cassée
2,3,Votre demande a été enregistrée.,votre demande a été enregistrée
3,4,Le produit est bon mais la livraison a été bea...,le produit est bon mais la livraison a été bea...
4,5,"Super, encore une panne de l'application.",super encore une panne de l'application
...,...,...,...
995,996,Je ne m'attendais pas à autant de satisfaction...,je ne m'attendais pas à autant de satisfaction...
996,997,"La livraison est fiable, rapide et soignée.",la livraison est fiable rapide et soignée
997,998,Je suis très satisfait de votre service dans l...,je suis très satisfait de votre service dans l...
998,999,"L'appli est bien conçue, intuitive et rapide.",l'appli est bien conçue intuitive et rapide


In [98]:
df["tokens"] = df["clean_avis"].apply(au.tokenize_text)
df["tokens_count"] = df["tokens"].apply(len)

print("\n=== Tokens ===")
df[["id", "clean_avis", "tokens", "tokens_count"]]


=== Tokens ===


,id,clean_avis,tokens,tokens_count
0,1,le service client a été très rapide et très pr...,"[le, service, client, a, été, très, rapide, et...",10
1,2,je suis très déçu ma commande est arrivée cassée,"[je, suis, très, déçu, ma, commande, est, arri...",9
2,3,votre demande a été enregistrée,"[votre, demande, a, été, enregistrée]",5
3,4,le produit est bon mais la livraison a été bea...,"[le, produit, est, bon, mais, la, livraison, a...",12
4,5,super encore une panne de l'application,"[super, encore, une, panne, de, l'application]",6
...,...,...,...,...
995,996,je ne m'attendais pas à autant de satisfaction...,"[je, ne, m'attendais, pas, à, autant, de, sati...",9
996,997,la livraison est fiable rapide et soignée,"[la, livraison, est, fiable, rapide, et, soignée]",7
997,998,je suis très satisfait de votre service dans l...,"[je, suis, très, satisfait, de, votre, service...",9
998,999,l'appli est bien conçue intuitive et rapide,"[l'appli, est, bien, conçue, intuitive, et, ra...",7


In [99]:
df["tokens_without_stopwords"] = df["tokens"].apply(au.remove_stop_words)
df["text_without_stopwords"] = df["tokens_without_stopwords"].apply(lambda tokens: " ".join(tokens))

print("\n=== Tokens sans stop words prudents ===")
df[["id", "tokens", "tokens_without_stopwords"]]


=== Tokens sans stop words prudents ===


,id,tokens,tokens_without_stopwords
0,1,"[le, service, client, a, été, très, rapide, et...","[service, client, très, rapide, très, professi..."
1,2,"[je, suis, très, déçu, ma, commande, est, arri...","[très, déçu, commande, arrivée, cassée]"
2,3,"[votre, demande, a, été, enregistrée]","[votre, demande, enregistrée]"
3,4,"[le, produit, est, bon, mais, la, livraison, a...","[produit, bon, mais, livraison, beaucoup, trop..."
4,5,"[super, encore, une, panne, de, l'application]","[super, encore, panne, l'application]"
...,...,...,...
995,996,"[je, ne, m'attendais, pas, à, autant, de, sati...","[ne, m'attendais, pas, autant, satisfaction, g..."
996,997,"[la, livraison, est, fiable, rapide, et, soignée]","[livraison, fiable, rapide, soignée]"
997,998,"[je, suis, très, satisfait, de, votre, service...","[très, satisfait, votre, service, l'ensemble]"
998,999,"[l'appli, est, bien, conçue, intuitive, et, ra...","[l'appli, bien, conçue, intuitive, rapide]"


In [100]:
df["lemmatized_tokens"] = df["tokens_without_stopwords"].apply(au.lemmatize_tokens)
df["processed_avis"] = df["lemmatized_tokens"].apply(lambda tokens: " ".join(tokens))

print("\n=== Texte final prétraité ===")
df[["id", "avis", "processed_avis"]]


=== Texte final prétraité ===


,id,avis,processed_avis
0,1,Le service client a été très rapide et très pr...,service client très rapide très professionnel
1,2,"Je suis très déçu, ma commande est arrivée cas...",très décevoir commande arriver casser
2,3,Votre demande a été enregistrée.,votre demande enregistrée
3,4,Le produit est bon mais la livraison a été bea...,produit bon mais livraison beaucoup trop longue
4,5,"Super, encore une panne de l'application.",super encore panne l'application
...,...,...,...
995,996,Je ne m'attendais pas à autant de satisfaction...,ne m'attendais pas autant satisfaction globale
996,997,"La livraison est fiable, rapide et soignée.",livraison fiable rapide soignée
997,998,Je suis très satisfait de votre service dans l...,très satisfaire votre service l'ensemble
998,999,"L'appli est bien conçue, intuitive et rapide.",l'appli bien conçue intuitive rapide


In [101]:
comparison_columns = [
    "id",
    "avis",
    "clean_avis",
    "text_without_stopwords",
    "processed_avis",
    "label"
]

print("\n=== Comparaison complète ===")
df[comparison_columns]


=== Comparaison complète ===


,id,avis,clean_avis,text_without_stopwords,processed_avis,label
0,1,Le service client a été très rapide et très pr...,le service client a été très rapide et très pr...,service client très rapide très professionnel,service client très rapide très professionnel,positif
1,2,"Je suis très déçu, ma commande est arrivée cas...",je suis très déçu ma commande est arrivée cassée,très déçu commande arrivée cassée,très décevoir commande arriver casser,négatif
2,3,Votre demande a été enregistrée.,votre demande a été enregistrée,votre demande enregistrée,votre demande enregistrée,neutre
3,4,Le produit est bon mais la livraison a été bea...,le produit est bon mais la livraison a été bea...,produit bon mais livraison beaucoup trop longue,produit bon mais livraison beaucoup trop longue,mitigé
4,5,"Super, encore une panne de l'application.",super encore une panne de l'application,super encore panne l'application,super encore panne l'application,négatif
...,...,...,...,...,...,...
995,996,Je ne m'attendais pas à autant de satisfaction...,je ne m'attendais pas à autant de satisfaction...,ne m'attendais pas autant satisfaction globale,ne m'attendais pas autant satisfaction globale,positif
996,997,"La livraison est fiable, rapide et soignée.",la livraison est fiable rapide et soignée,livraison fiable rapide soignée,livraison fiable rapide soignée,positif
997,998,Je suis très satisfait de votre service dans l...,je suis très satisfait de votre service dans l...,très satisfait votre service l'ensemble,très satisfaire votre service l'ensemble,positif
998,999,"L'appli est bien conçue, intuitive et rapide.",l'appli est bien conçue intuitive et rapide,l'appli bien conçue intuitive rapide,l'appli bien conçue intuitive rapide,positif


In [102]:
output_columns = [
    "id",
    "avis",
    "label",
    "categorie",
    "difficulte",
    "clean_avis",
    "tokens_count",
    "text_without_stopwords",
    "processed_avis"
]

df[output_columns].to_csv("../Data/Processed/Avis_client_processed.csv", index=False, encoding="utf-8")

print("\n=== Export terminé ===")
print(f"Fichier généré : {"../Data/Processed/Avis_client_processed.csv"}")


=== Export terminé ===
Fichier généré : ../Data/Processed/Avis_client_processed.csv
